# PR #49343 validation — EAGLE-3 2048-crash fix (A100)

Validates [vllm-project/vllm#49343](https://github.com/vllm-project/vllm/pull/49343)
against the original [#48894](https://github.com/vllm-project/vllm/issues/48894) repro
(EAGLE-3 + compiled mode + ~7.4k-token RAG prompt → device-side assert at position 2048).

**Part A** — serve with the PR applied: capture (a) the `Overriding draft model
max_position_embeddings` startup line, (b) request completes with no assert, (c) tau.
**Part B** — same build, fix stripped (`git checkout main -- vllm/config/speculative.py`),
same prompt: capture the crash coming back (same traceback shape as #48894).

Conventions inherited from `phase3c_diagnostics_40gb.ipynb`: isolated venv for vLLM,
harness-built RAG prompt (same params/seed as the crashing diag cells), tau from the
same Prometheus counters via `harness.metrics`, process-group-safe teardown. The full
two-signal stall watchdog is a sweep tool; this manual launch uses the simplified form
(process-death check + log-growth stall signal) since there is no HF-cache download
phase worth watching separately.

Launch flags match the Phase-3c crashing cells **minus** `--compilation-config
{"cudagraph_mode": "NONE"}`: that probe flag was for bisecting CUDA graphs vs compile,
and default compiled mode also crashed (Phase 3b). Default mode avoids any flag-name
drift between vLLM 0.24.0 and main.

Runtime: ~30–45 min total (venv build ~8 min, model download ~5 min, compile ~5 min/launch).


In [1]:
# 1. GPU check — any A100 is fine here (crash/no-crash + tau are not memory-bound;
# tau comparability to the 40GB records holds on 80GB too).
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import subprocess
gpu = subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert "A100" in gpu, "Not an A100 - reconnect until Colab honors the selection."
print("GPU OK:", gpu)


NVIDIA A100-SXM4-40GB, 40960 MiB
GPU OK: NVIDIA A100-SXM4-40GB


In [2]:
# 2. Study repo + Spec-Bench (the RAG documents the crashing prompt is built from)
%cd /content
!git clone https://github.com/manojarulmurugan/SpecDecoding-Study-vLLM-SGLang.git repo 2>/dev/null || (cd repo && git pull)
%cd /content/repo
!git clone --depth 1 https://github.com/hemingkx/Spec-Bench.git external/Spec-Bench 2>/dev/null || echo "Spec-Bench already present"
!test -f external/Spec-Bench/data/spec_bench/question.jsonl && echo "RAG data OK"


/content
/content/repo
RAG data OK


In [3]:
# 3. vLLM at PR #49343 in an isolated venv (python-only PR -> precompiled binaries OK).
# Plain git (no gh auth needed): fetch the PR head into a local branch.
%cd /content
!git clone https://github.com/vllm-project/vllm.git vllm 2>/dev/null || echo "vllm clone already present"
%cd /content/vllm
!git fetch origin pull/49343/head:pr49343 && git checkout pr49343
!git log --oneline -3
PR_SHA = subprocess.run(["git","rev-parse","--short","HEAD"], capture_output=True, text=True,
                        cwd="/content/vllm").stdout.strip()
print("PR head:", PR_SHA, " <-- copy this back (goes in the review comment)")
# The fix must be present before we build:
!grep -n "_maybe_override_draft_max_position_embeddings" vllm/config/speculative.py | head -3

%pip install -q virtualenv
!virtualenv -q /content/vllm_pr_env
!/content/vllm_pr_env/bin/pip install -q --upgrade pip
# ~8 min: downloads the prebuilt kernels for the PR's merge-base commit, installs
# the PR's python on top (editable -> Part B's git checkout swap takes effect on restart).
!cd /content/vllm && VLLM_USE_PRECOMPILED=1 /content/vllm_pr_env/bin/pip install -q -e .
!/content/vllm_pr_env/bin/python -c "import vllm; print('vllm', vllm.__version__)"


/content
/content/vllm
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 29 (delta 17), reused 17 (delta 17), pack-reused 8 (from 2)
Unpacking objects: 100% (29/29), 36.70 KiB | 736.00 KiB/s, done.
From https://github.com/vllm-project/vllm
 * [new ref]             refs/pull/49343/head -> pr49343
Switched to branch 'pr49343'
f5a7f2eda (HEAD -> pr49343) Merge branch 'main' into fix-eagle-draft-max-position-embeddings
9df2f9123 [Renderer] Offload derender CPU work to renderer thread pool (#49396)
387189c42 [ROCm] Remove redundant AITER fused_qk_rmsnorm probe (avoids config-time HIP init) (#47992)
PR head: f5a7f2eda  <-- copy this back (goes in the review comment)
917:                    SpeculativeConfig._maybe_override_draft_max_position_embeddings(
1143:    def _maybe_override_draft_max_position_embeddings(
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 77.5 MB/s eta 0:00:00
   ━━━━━━

**If cell 3's `VLLM_USE_PRECOMPILED` install fails** (no wheel for the merge-base),
fall back to: `pip install -U vllm --pre --extra-index-url https://wheels.vllm.ai/nightly`
into the venv, then copy the PR's two changed source files
(`vllm/config/speculative.py`, `vllm/v1/spec_decode/llm_base_proposer.py`) from
`/content/vllm` over the installed package in
`/content/vllm_pr_env/lib/python*/site-packages/vllm/`. Note in what you report back
that the fallback was used — the nightly base may differ slightly from the PR's base.


In [5]:
# 4. HF auth (Llama-3.1 is gated). Token must reach BOTH this python (tokenizer)
# and the server env (checkpoint download).
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    from getpass import getpass
    os.environ["HF_TOKEN"] = getpass("HF token: ")
from huggingface_hub import login
login(os.environ["HF_TOKEN"])


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [6]:
# 5. Build the repro prompts with the project's own workload code — identical
# template/seed/doc-sizing to the crashing Phase-3c cells (doc_target_tokens=7400,
# prompt_token_budget=7900, seed=1234). num_requests reduced 24 -> 8: enough for a
# stable tau read; the crash itself fires on the first request's prefill.
import sys
sys.path.insert(0, "/content/repo")
from transformers import AutoTokenizer
from harness.workloads.rag_shared_prefix import RagSharedPrefixWorkload

TOK = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
params = dict(
    num_requests=8, max_new_tokens=256, prefix_overlap="low",
    doc_target_tokens=7400, tokenizer_model="meta-llama/Llama-3.1-8B-Instruct",
    prompt_token_budget=7900, tokenizer=TOK,
    spec_bench_file="/content/repo/external/Spec-Bench/data/spec_bench/question.jsonl",
)
ITEMS = RagSharedPrefixWorkload(params, seed=1234).build()
lens = [len(TOK.encode(it.prompt)) for it in ITEMS]
print(len(ITEMS), "prompts; token lengths:", lens)
assert all(l > 2048 for l in lens), "prompts must exceed 2048 tokens to exercise the bug"


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


8 prompts; token lengths: [7435, 7435, 7438, 7438, 7435, 7435, 7435, 7436]


In [7]:
# 6. Server lifecycle (adapter-derived: start_new_session launch, /health polling with
# death + log-stall detection, killpg teardown). Same command for Part A and Part B.
import json, os, signal, subprocess, time, requests

VENV = "/content/vllm_pr_env"
BASE = "http://127.0.0.1:8000"
LOGDIR = "/content/logs"; os.makedirs(LOGDIR, exist_ok=True)

LAUNCH_CMD = [
    f"{VENV}/bin/vllm", "serve", "meta-llama/Llama-3.1-8B-Instruct",
    "--host", "127.0.0.1", "--port", "8000",
    "--dtype", "float16", "--gpu-memory-utilization", "0.85",
    "--max-model-len", "8192", "--seed", "1234",
    "--no-enable-prefix-caching",
    "--max-num-batched-tokens", "8192",
    "--speculative-config", json.dumps({
        "method": "eagle3",
        "model": "yuhuili/EAGLE3-LLaMA3.1-Instruct-8B",
        "num_speculative_tokens": 5,
    }),
]   # NOTE: no --enforce-eager (compiled mode is the point), stock draft checkpoint.

def launch(tag):
    log_path = f"{LOGDIR}/{tag}.log"
    log_fh = open(log_path, "w")
    env = dict(os.environ, PATH=f"{VENV}/bin:" + os.environ["PATH"])
    proc = subprocess.Popen(LAUNCH_CMD, stdout=log_fh, stderr=subprocess.STDOUT,
                            start_new_session=True, env=env)
    return proc, log_path

def tail(log_path, n=40):
    with open(log_path, errors="replace") as fh:
        return "".join(fh.readlines()[-n:])

def wait_ready(proc, log_path, timeout_s=2400, stall_s=600):
    t0 = time.monotonic(); last_size, last_growth = -1, time.monotonic()
    while time.monotonic() - t0 < timeout_s:
        if proc.poll() is not None:
            raise RuntimeError("server DIED during startup:\n" + tail(log_path))
        try:
            if requests.get(BASE + "/health", timeout=5).status_code == 200:
                return
        except requests.RequestException:
            pass
        size = os.path.getsize(log_path)
        if size != last_size:
            last_size, last_growth = size, time.monotonic()
        elif time.monotonic() - last_growth > stall_s:
            raise RuntimeError(f"server STALLED (no /health, log frozen {stall_s}s):\n"
                               + tail(log_path))
        time.sleep(5)
    raise RuntimeError(f"server not ready in {timeout_s}s:\n" + tail(log_path))

def teardown(proc):
    try:
        os.killpg(proc.pid, signal.SIGTERM)
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            os.killpg(proc.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass  # already dead (expected after Part B's crash)
    time.sleep(10)  # let the GPU drain before the next launch

OVERRIDE_PAT = "Overriding draft model max_position_embeddings"
CRASH_PATS = ("device-side assert", "AcceleratorError", "index out of bounds")

def grep_log(log_path, pats):
    hits = []
    with open(log_path, errors="replace") as fh:
        for i, line in enumerate(fh):
            if any(p in line for p in pats):
                hits.append((i + 1, line.rstrip()))
    return hits

def completion(prompt, timeout=1800):
    r = requests.post(BASE + "/v1/completions", timeout=timeout, json={
        "model": "meta-llama/Llama-3.1-8B-Instruct",
        "prompt": prompt, "max_tokens": 256, "temperature": 0})
    r.raise_for_status()
    return r.json()["choices"][0]["text"]


In [8]:
# 7. PART A — the fix. Expect: override line at startup, all 8 requests complete,
# no assert in the log, tau reported from the same counters the study used.
from harness.metrics import parse_prometheus_text, spec_decode_stats

proc, LOG_A = launch("partA_fix")
wait_ready(proc, LOG_A)

ov = grep_log(LOG_A, (OVERRIDE_PAT,))
assert ov, "override line NOT found - the fix did not fire; STOP and report the log"
print("OVERRIDE LINE (copy back verbatim):")
print("   ", ov[0][1])

m0 = parse_prometheus_text(requests.get(BASE + "/metrics", timeout=10).text)
t0 = time.time()
for i, it in enumerate(ITEMS):
    out = completion(it.prompt)
    print("req %d ok (%d chars): %r..." % (i, len(out), out[:60]))
m1 = parse_prometheus_text(requests.get(BASE + "/metrics", timeout=10).text)

stats = spec_decode_stats(m0, m1)
crash = grep_log(LOG_A, CRASH_PATS)
teardown(proc)

summary_a = {
    "pr_head": PR_SHA, "gpu": gpu, "n_requests": len(ITEMS),
    "prompt_token_lengths": lens, "wall_s": round(time.time() - t0, 1),
    "override_line": ov[0][1], "crash_lines": crash, "spec_stats": stats,
}
print("\n=========== PART A VERDICT ===========")
print(json.dumps(summary_a, indent=2))
print("tau = %.4f  (drafts=%d, draft_tokens=%d, accepted=%d)" % (
    stats["accepted_length_tau"], stats["num_drafts"],
    stats["num_draft_tokens"], stats["num_accepted_tokens"]))
print("no-crash check:", "PASS (no assert patterns in log)" if not crash else "FAIL")
with open(f"{LOGDIR}/partA_summary.json", "w") as fh:
    json.dump(summary_a, fh, indent=2)


OVERRIDE LINE (copy back verbatim):
    (APIServer pid=2805) INFO 07-22 10:40:28 [speculative.py:1167] Overriding draft model max_position_embeddings from 2048 to the target model's max_model_len (8192); EAGLE drafts share the target's positional space.
req 0 ok (1047 chars): " George Harrison, James Ray, Shakin' Stevens, Lee Matthews, "...
req 1 ok (1336 chars): ' The Allies commenced an attack of their own in Egypt, dislo'...
req 2 ok (1157 chars): " Trick-or-Treat for UNICEF money goes to support UNICEF's gl"...
req 3 ok (1188 chars): ' The document does not mention the lowest barometric pressur'...
req 4 ok (1368 chars): ' There is no information in the provided text about the lead'...
req 5 ok (1190 chars): ' The term "prime minister" was first used in 1721 when Rober'...
req 6 ok (1383 chars): ' Forests provide various benefits to humans, but they can al'...
req 7 ok (1078 chars): " \nI don't have the information to answer that question. The "...

=========== PART A VERDICT =====

In [9]:
# 8. PART B — the control. Strip the fix from the SAME editable install, restart,
# re-fire the identical first prompt, expect the #48894 assert back.
!cd /content/vllm && git checkout main -- vllm/config/speculative.py
!grep -c "_maybe_override_draft_max_position_embeddings" /content/vllm/vllm/config/speculative.py || echo "fix stripped OK"

proc, LOG_B = launch("partB_control")
wait_ready(proc, LOG_B)
assert not grep_log(LOG_B, (OVERRIDE_PAT,)), "override line present - strip failed; STOP"
print("override line absent, as expected. Firing the crash prompt...")

err = None
try:
    out = completion(ITEMS[0].prompt, timeout=600)
    print("UNEXPECTED: request completed:", out[:80])
except Exception as e:
    err = repr(e)
    print("request failed (expected):", err[:200])

deadline = time.time() + 120   # give the engine a moment to flush the traceback
crash = []
while time.time() < deadline and not crash:
    time.sleep(5); crash = grep_log(LOG_B, CRASH_PATS)
teardown(proc)

print("\n=========== PART B VERDICT ===========")
print("crash reproduced:", "YES" if crash else "NO  <-- if NO, STOP and report full log")
for ln, line in crash[:6]:
    print("  L%d: %s" % (ln, line))
# 30-line excerpt around the first crash line (this is what gets pasted in the review):
if crash:
    first = crash[0][0]
    with open(LOG_B, errors="replace") as fh:
        all_lines = fh.readlines()
    excerpt = "".join(all_lines[max(0, first - 5):first + 25])
    print("\n----- EXCERPT (copy back) -----\n" + excerpt)
    with open(f"{LOGDIR}/partB_excerpt.txt", "w") as fh:
        fh.write(excerpt)

# leave the vllm tree back on the PR branch
!cd /content/vllm && git checkout pr49343 -- vllm/config/speculative.py


0
fix stripped OK
override line absent, as expected. Firing the crash prompt...
request failed (expected): HTTPError('500 Server Error: Internal Server Error for url: http://127.0.0.1:8000/v1/completions')

=========== PART B VERDICT ===========
crash reproduced: YES
  L180: /root/.cache/vllm/torch_compile_cache/torch_aot_compile/5843ad2245b417ab0446d1d490bbc35dd6ea6f2015ef90ac38065e9a8eff70e3/inductor_cache/g6/cg6rzqiekvqzulfds3s7cm3nbiqids7ahdew7k3qiw6cwk7hooud.py:45: unknown: block: [2259,0,0], thread: [96,0,0] Assertion `index out of bounds: 0 <= tl.broadcast_to(tmp10, [XBLOCK]) < 2048` failed.
  L181: /root/.cache/vllm/torch_compile_cache/torch_aot_compile/5843ad2245b417ab0446d1d490bbc35dd6ea6f2015ef90ac38065e9a8eff70e3/inductor_cache/g6/cg6rzqiekvqzulfds3s7cm3nbiqids7ahdew7k3qiw6cwk7hooud.py:45: unknown: block: [2259,0,0], thread: [97,0,0] Assertion `index out of bounds: 0 <= tl.broadcast_to(tmp10, [XBLOCK]) < 2048` failed.
  L182: /root/.cache/vllm/torch_compile_cache/torch_aot_

In [10]:
# 9. Bundle everything to bring back for review.
import zipfile
report = f"""PR49343 VALIDATION BUNDLE
pr_head: {PR_SHA}
gpu: {gpu}
Part A summary: see partA_summary.json
Part B excerpt: see partB_excerpt.txt
Full logs: partA_fix.log, partB_control.log
"""
open(f"{LOGDIR}/README.txt", "w").write(report)
with zipfile.ZipFile("/content/pr49343_validation_bundle.zip", "w") as z:
    for name in os.listdir(LOGDIR):
        z.write(os.path.join(LOGDIR, name), name)
print("bundle at /content/pr49343_validation_bundle.zip - download it")
print(report)


bundle at /content/pr49343_validation_bundle.zip - download it
PR49343 VALIDATION BUNDLE
pr_head: f5a7f2eda
gpu: NVIDIA A100-SXM4-40GB
Part A summary: see partA_summary.json
Part B excerpt: see partB_excerpt.txt
Full logs: partA_fix.log, partB_control.log

